# Knowledge Graph Pipeline: End-to-End Data Ingestion & Labeling

This notebook walks through the **complete KG pipeline** — from pulling raw Telegram messages
to a fully populated knowledge graph with heat-based phase classification.

Each section executes one pipeline stage against **live infrastructure** (Postgres, Redis, Pinecone, OpenAI),
with observability into intermediate data, timing, and counts at every step.

## Pipeline Stages

| # | Stage | Writes to |
|---|-------|-----------|
| 1 | Environment Setup & Connections | — |
| 2 | Telegram Client & Channel Discovery | Postgres (channel_profiles) |
| 3 | Message Ingestion | Postgres (raw_messages), Redis stream |
| 4 | Translation | Postgres (raw_messages.english_text), OpenAI API |
| 5 | Story Segmentation | Postgres (story_units) |
| 6 | Story Embeddings | Pinecone (story index), OpenAI API |
| 7 | Semantic Extraction Preview | OpenAI API (read-only preview) |
| 8 | Node Resolution & Assignment | Postgres (nodes, story_nodes, story_semantics), Pinecone (centroids) |
| 9 | Projection (Relations & Stats) | Postgres (node_relations, theme_daily_stats) |
| 10 | Event Hierarchy | Postgres (nodes.parent_node_id) |
| 11 | Heat & Phase Classification | Postgres (node_heat_view) |
| 12 | Query & Inspect | — (read-only) |
| 13 | Pipeline Summary | — |
| 14 | Cleanup | — |

## Prerequisites

- A populated `.env` file at the project root with: `TG_API_ID`, `TG_API_HASH`, `TG_PHONE`,
  `SESSION_PATH`, `DATABASE_URL`, `REDIS_URL`, `OPENAI_API_KEY`, `PINECONE_API_KEY`,
  `PINECONE_INDEX_STORY`, `PINECONE_INDEX_THEME`, `PINECONE_INDEX_EVENT`
- Running Postgres and Redis instances (see `docker-compose.yml`)
- An authenticated Telegram session (run `telegram-scraper login` first)
- Run this notebook from the **project root** directory

> **Warning:** This notebook writes real data to all connected stores.
> Use the `MESSAGE_LIMIT` parameter to control scope.

---
## 1. Environment Setup & Connections

Load configuration from `.env`, validate that all required services are reachable,
and build the infrastructure objects (repository, stream, embedder, extractor, translator, vector store)
using the factory functions in `kg/runtime.py`.

In [14]:
import sys
import os
import time
from pathlib import Path

# Ensure the project src/ is importable when running from notebooks/ or project root.
project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    # We're inside a subdirectory (e.g. notebooks/) — go up one level.
    project_root = project_root.parent
    os.chdir(project_root)
src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# nest_asyncio allows top-level await in Jupyter even on older kernels.
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass  # Modern IPyKernel 6+ handles top-level await natively.

import pandas as pd
from IPython.display import display, JSON, Markdown

# Widen DataFrame display for readable text previews.
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 60)

print(f"Project root: {project_root}")

Project root: /Users/lfpmb/Documents/telegram-twitter-scraper


In [15]:
# Load configuration from .env and validate every subsystem.
from telegram_scraper.kg.config import KGSettings
from telegram_scraper.config import Settings

kg_settings = KGSettings.load(".env")
app_settings = Settings.load(".env")

# These raise ConfigError with a clear message if any env var is missing.
kg_settings.require_database()
kg_settings.require_stream()
kg_settings.require_vector_store()
kg_settings.require_embeddings()
kg_settings.require_semantic_extraction()
kg_settings.require_translation()
app_settings.require_credentials()

print("All configuration validated.")
print(f"  Database:        {kg_settings.database_url.split('@')[-1] if '@' in kg_settings.database_url else '(set)'}")
print(f"  Redis:           {kg_settings.redis_url}")
print(f"  Embedding model: {kg_settings.embedding_model}")
print(f"  Semantic model:  {kg_settings.semantic_model}")
print(f"  Translation:     {kg_settings.translation_model}")
print(f"  Pinecone story:  {kg_settings.pinecone_index_story}")
print(f"  Pinecone theme:  {kg_settings.pinecone_index_theme}")
print(f"  Pinecone event:  {kg_settings.pinecone_index_event}")

All configuration validated.
  Database:        localhost:5433/telegram_kg
  Redis:           redis://localhost:6379/0
  Embedding model: text-embedding-3-small
  Semantic model:  gpt-5-mini
  Translation:     gpt-5-mini
  Pinecone story:  story-embeddings
  Pinecone theme:  topic-centroids
  Pinecone event:  event-centroids


In [16]:
# Build all infrastructure objects via the canonical factory functions.
# Each builder reads credentials from KGSettings and returns a ready-to-use instance.
from telegram_scraper.kg.runtime import (
    build_repository,
    build_stream,
    build_embedder,
    build_semantic_extractor,
    build_message_translator,
    build_vector_store,
)

repository = build_repository(kg_settings)     # PostgresStoryRepository
stream = build_stream(kg_settings)              # RedisRawMessageStream
embedder = build_embedder(kg_settings)          # OpenAIEmbedder
extractor = build_semantic_extractor(kg_settings)  # OpenAISemanticExtractor
translator = build_message_translator(kg_settings) # OpenAIMessageTranslator
vector_store = build_vector_store(kg_settings)     # PineconeVectorStore

# Ensure database schema and Redis consumer group exist.
repository.ensure_schema()
stream.ensure_group()

# Quick smoke test: how many channels are already in the DB?
existing_channels = repository.list_channels()
print(f"Connections established. Existing channels in DB: {len(existing_channels)}")
if existing_channels:
    for ch in existing_channels:
        print(f"  - {ch.channel_title} (id={ch.channel_id}, stories={ch.story_count})")

Connections established. Existing channels in DB: 5
  - Amir Tsarfati (id=-1001361890342, stories=1171)
  - IRNA News Agency 🇮🇷 (id=-1001710170626, stories=775)
  - Press TV (id=-1001006487902, stories=2634)
  - СОЛОВЬЁВ (id=-1001315735637, stories=2235)
  - 🇮🇷Defender IRAN🇮🇷 (id=-1001020219579, stories=117)


---
## 2. Telegram Client & Channel Discovery

Connect to Telegram using the authenticated session, discover all available dialogs,
and filter down to **channels** (the primary data source for the KG pipeline).
You'll pick one channel and set a message limit for this run.

In [17]:
# Create and connect the Telegram client.
# This reuses the session file created by `telegram-scraper login`.
from telegram_scraper.telegram_client import TelegramAccountClient

telegram_client = TelegramAccountClient(
    api_id=app_settings.api_id or 0,
    api_hash=app_settings.api_hash,
    session_path=app_settings.session_path,
    output_root=app_settings.output_root,
    phone=app_settings.phone,
)
await telegram_client.connect()
print("Telegram client connected.")

Telegram client connected.


Server closed the connection: 0 bytes read on a total of 8 expected bytes
Error executing high-level request after reconnect: <class 'sqlite3.OperationalError'>: database is locked


In [18]:
# Discover all dialogs and filter to channels.
from telegram_scraper.chat_discovery import discover_chats, filter_chats
from telegram_scraper.models import ChatType

dialogs = await telegram_client.get_dialogs()
all_chats = discover_chats(dialogs)
channels = filter_chats(
    list(all_chats),
    chat_types=(ChatType.CHANNEL,),
    include_chats=app_settings.include_chats,
    exclude_chats=app_settings.exclude_chats,
)

channel_df = pd.DataFrame([
    {
        "#": i,
        "chat_id": c.chat_id,
        "title": c.title,
        "username": c.username or "",
        "slug": c.slug,
    }
    for i, c in enumerate(channels)
])
print(f"Found {len(channels)} channels (after include/exclude filters):")
display(channel_df)

Found 23 channels (after include/exclude filters):


,#,chat_id,title,username,slug
0,0,-1001361890342,Amir Tsarfati,beholdisraelchannel,beholdisraelchannel
1,1,-1001702330601,Jewish Breaking News - JBN,JewishBreakingNewsTelegram,jewishbreakingnewstelegram
2,2,-1001006487902,Press TV,presstv,presstv
3,3,-1001170925964,محمد مهدی بابایی War news,m_Mahdibaba,m-mahdibaba
4,4,-1001676275372,DD Geopolitics,DDGeopolitics,ddgeopolitics
5,5,-1001860107178,Geopolitics Prime | Iran War Updates,geopolitics_prime,geopolitics-prime
6,6,-1001607417124,The Times of Israel,TheTimesOfIsrael2022,thetimesofisrael2022
7,7,-1001421185290,Israel Realtime,Israel_Realtime_Updates,israel-realtime-updates
8,8,-1001425249914,🏴 Iranian Militarism 🏴,Iranian_Militarism,iranian-militarism
9,9,-1001894359082,Middle East Eye,MiddleEastEye_TG,middleeasteye-tg


In [20]:
# ============================================================
# >>> CONFIGURE YOUR RUN HERE <<<
# Set the channel index (from the table above) and message limit.
# ============================================================

CHANNEL_INDEX = 2       # Change this to pick a different channel.
MESSAGE_LIMIT = 100     # Number of messages to ingest (most recent first).

# ============================================================

selected_chat = channels[CHANNEL_INDEX]
print(f"Selected channel: {selected_chat.title}")
print(f"  Chat ID:  {selected_chat.chat_id}")
print(f"  Username: {selected_chat.username or '(none)'}")
print(f"  Slug:     {selected_chat.slug}")
print(f"  Limit:    {MESSAGE_LIMIT} messages")

Selected channel: Press TV
  Chat ID:  -1001006487902
  Username: presstv
  Slug:     presstv
  Limit:    100 messages


In [21]:
# Sync the channel profile into Postgres.
# The profile stores segmentation settings (time gap, delimiter patterns, merge threshold)
# and is created with sensible defaults if it doesn't exist yet.
from telegram_scraper.kg.services import KGProfileService
from telegram_scraper.kg.segmentation import default_channel_profile

profile_service = KGProfileService(repository)
profile_service.sync_chat_metadata([selected_chat])

profile = repository.get_channel_profile(selected_chat.chat_id) or default_channel_profile(selected_chat.chat_id)

print(f"Channel profile for {selected_chat.title}:")
print(f"  Time gap (story break):  {profile.time_gap_minutes} min")
print(f"  Lookback story count:   {profile.lookback_story_count}")
print(f"  Media group window:     {profile.media_group_window_seconds}s")
print(f"  Similarity merge thresh: {profile.similarity_merge_threshold}")
print(f"  Delimiter patterns:      {len(profile.delimiter_patterns)}")
for dp in profile.delimiter_patterns:
    print(f"    - [{dp.kind}] {dp.pattern}")

Channel profile for Press TV:
  Time gap (story break):  10 min
  Lookback story count:   5
  Media group window:     60s
  Similarity merge thresh: 0.7
  Delimiter patterns:      0


---
## 3. Message Ingestion

Fetch messages from the selected Telegram channel, normalize them into `RawMessage` objects,
persist to Postgres (`raw_messages` table), and push to the Redis stream for downstream consumers.

**Normalization** extracts: channel_id, message_id, timestamp, text, sender info,
media references, forwarding metadata, and the raw JSON envelope.

In [22]:
from telegram_scraper.kg.normalization import normalize_message_record

t0 = time.time()
raw_messages = []
async for envelope in telegram_client.iter_message_envelopes(
    selected_chat,
    limit=MESSAGE_LIMIT,
    reverse=False,  # newest first — fetch the most recent messages
):
    raw_msg = normalize_message_record(envelope.record, raw_json=envelope.raw_json)
    raw_messages.append(raw_msg)

# Sort chronologically for downstream segmentation.
raw_messages.sort(key=lambda m: (m.timestamp, m.message_id))

elapsed_fetch = time.time() - t0
print(f"Fetched {len(raw_messages)} messages in {elapsed_fetch:.1f}s")

# Persist to Postgres.
t0 = time.time()
repository.upsert_raw_messages(raw_messages)
elapsed_db = time.time() - t0
print(f"Upserted to Postgres in {elapsed_db:.1f}s")

# Push to Redis stream (maintains parity with the production kg-backfill command).
t0 = time.time()
for msg in raw_messages:
    stream.add(msg)
elapsed_redis = time.time() - t0
print(f"Streamed to Redis in {elapsed_redis:.1f}s")

Fetched 100 messages in 3.5s
Upserted to Postgres in 0.3s
Streamed to Redis in 0.5s


In [23]:
# Inspect a sample of ingested messages.
msg_df = pd.DataFrame([
    {
        "message_id": m.message_id,
        "timestamp": m.timestamp.strftime("%Y-%m-%d %H:%M") if m.timestamp else "",
        "sender": m.sender_name or "",
        "text_preview": (m.text or "")[:120],
        "has_media": bool(m.media_refs),
        "forwarded": m.forwarded_from is not None,
    }
    for m in raw_messages[:25]
])

print(f"Showing first {len(msg_df)} of {len(raw_messages)} messages:")
display(msg_df)

# Summary stats.
n_with_text = sum(1 for m in raw_messages if (m.text or "").strip())
n_with_media = sum(1 for m in raw_messages if m.media_refs)
n_forwarded = sum(1 for m in raw_messages if m.forwarded_from is not None)
print(f"\nStats: {n_with_text} with text, {n_with_media} with media, {n_forwarded} forwarded")

Showing first 25 of 100 messages:


,message_id,timestamp,sender,text_preview,has_media,forwarded
0,185268,2026-04-13 11:10,presstv,Iran warns any attack on its ports will make every Persian Gulf port insecure\n\nIran's military has warned that any threa,True,False
1,185269,2026-04-13 11:20,presstv,"The commander of the Quds Force of the Islamic Revolution Guard Corps, Brigadier General Esmaeil Qaani, warned that both",False,False
2,185270,2026-04-13 11:33,presstv,Spokesman of the Khatam Al-Anbiya Central Headquarters warns that the security in the Persian Gulf and Sea of Oman is ei,False,False
3,185271,2026-04-13 11:47,presstv,🔴 Leader of the Islamic Movement in Nigeria Sheikh Zakzaky:\n\n🔺 The Strait of Hormuz and Bab al-Mandab are not unclaimed,True,False
4,185272,2026-04-13 11:53,presstv,US-Israeli aggression: 960 people rescued alive from under rubble in Tehran \n\nThe Iranian Red Crescent Society (IRCS) s,True,False
5,185273,2026-04-13 11:58,presstv,"‘Unhinged & unchristian’: Global outrage erupts after Trump attacks Pope, poses as Jesus\n\nA massive firestorm of interna",True,False
6,185274,2026-04-13 12:05,presstv,Tehran police arrested a suspect in a raid after receiving reports of illegal weapons storage in the eastern part of the,False,False
7,185275,2026-04-13 12:17,presstv,Iran restores key rail links after damage caused by US-Israeli assaults\n\nIran has restored train services in East Azarba,False,False
8,185276,2026-04-13 12:23,presstv,"Children’s shoes were displayed at Amsterdam’s Dam Square during a memorial protest on April 12, honoring Gaza’s childre",True,False
9,185277,2026-04-13 12:23,presstv,,True,False



Stats: 73 with text, 77 with media, 4 forwarded


---
## 4. Translation

The `OpenAIMessageTranslator` processes each raw message:

1. **Already English?** Detected via regex (>90% ASCII alpha) — passes through with `source_language='en'`.
2. **Empty text?** Marked with `source_language='und'` (undetermined).
3. **Non-English?** Sent to OpenAI for translation. Batched for efficiency.

Translated text is stored in `raw_messages.english_text` — the original is never modified.

In [24]:
t0 = time.time()
translated_messages = translator.translate_messages(raw_messages)
repository.save_raw_message_translations(translated_messages)
elapsed_translate = time.time() - t0

# Language distribution.
lang_series = pd.Series(
    [m.source_language or "unknown" for m in translated_messages]
).value_counts()

print(f"Translated {len(translated_messages)} messages in {elapsed_translate:.1f}s")
print(f"\nLanguage distribution:")
display(lang_series.to_frame("count"))

Translated 100 messages in 0.1s

Language distribution:


,count
en,73
und,27


In [25]:
# Show before/after for messages that were actually translated (not English, not empty).
actually_translated = [
    m for m in translated_messages
    if m.source_language not in ("en", "und", None)
]

if actually_translated:
    print(f"Showing {min(5, len(actually_translated))} of {len(actually_translated)} translated messages:\n")
    for m in actually_translated[:5]:
        print(f"--- message {m.message_id} (lang={m.source_language}) ---")
        print(f"  Original:   {(m.text or '')[:200]}")
        print(f"  Translated: {(m.english_text or '')[:200]}")
        print()
else:
    print("All messages were already English (or empty). No translation needed.")

All messages were already English (or empty). No translation needed.


---
## 5. Story Segmentation (Semantic Grouping)

Messages are grouped into stories by **embedding similarity** — topically related messages
become a single story regardless of when they were posted. This correctly groups things like
multiple quotes from the same speech, or follow-up posts about the same event, without
merging unrelated posts that happen to be close in time.

**Algorithm:**
1. Embed every non-empty message individually.
2. Walk through messages chronologically.
3. For each message, check cosine similarity against every existing story cluster
   (using the centroid — average of message embeddings in the cluster).
4. If the best match exceeds `SIMILARITY_THRESHOLD`, attach to that cluster.
5. Otherwise, start a new cluster.

This approach is media-channel-appropriate: it respects that each post is a standalone
unit, but still rolls up multi-part coverage of the same event.

In [ ]:
# ============================================================
# >>> TUNE SEMANTIC GROUPING <<<
# Higher threshold = stricter grouping (fewer merges).
# 0.65-0.75 is a good range for news-style messages.
# ============================================================
SIMILARITY_THRESHOLD = 0.70
# ============================================================

from telegram_scraper.kg.segmentation import StorySegmenter
from telegram_scraper.kg.extraction import preferred_story_text, safe_story_text
from telegram_scraper.kg.math_utils import cosine_similarity, average_vectors

segmenter = StorySegmenter()

# Step 1: Keep only messages with real text, sorted chronologically.
non_empty = [
    m for m in translated_messages
    if (m.english_text or m.text or "").strip()
]
non_empty.sort(key=lambda m: (m.timestamp, m.message_id))

# Step 2: Embed every message individually.
t0 = time.time()
per_message_texts = [
    safe_story_text((m.english_text or m.text or "").strip(), max_chars=kg_settings.semantic_max_chars)
    for m in non_empty
]
per_message_embeddings = embedder.embed_texts(per_message_texts) if per_message_texts else []
elapsed_msg_embed = time.time() - t0
print(f"Embedded {len(non_empty)} messages in {elapsed_msg_embed:.1f}s")

# Step 3: Cluster by similarity against existing cluster centroids.
clusters: list[list] = []          # list of message lists
centroids: list[list[float]] = []  # running mean embedding per cluster

for msg, emb in zip(non_empty, per_message_embeddings):
    best_idx, best_sim = -1, 0.0
    for idx, centroid in enumerate(centroids):
        sim = cosine_similarity(centroid, emb)
        if sim > best_sim:
            best_sim, best_idx = sim, idx
    if best_idx >= 0 and best_sim >= SIMILARITY_THRESHOLD:
        clusters[best_idx].append(msg)
        centroids[best_idx] = average_vectors([centroids[best_idx], emb])
    else:
        clusters.append([msg])
        centroids.append(list(emb))

# Step 4: Each cluster becomes a StoryUnit (sorted by the first message's timestamp).
for cluster in clusters:
    cluster.sort(key=lambda m: (m.timestamp, m.message_id))
clusters.sort(key=lambda c: (c[0].timestamp, c[0].message_id))

t0 = time.time()
stories = [segmenter.create_story(cluster, profile=profile) for cluster in clusters]
repository.save_story_units(stories)
elapsed_segment = time.time() - t0

multi_message_stories = sum(1 for c in clusters if len(c) > 1)
print(f"\nCreated {len(stories)} stories from {len(non_empty)} messages (threshold={SIMILARITY_THRESHOLD})")
print(f"  Multi-message stories: {multi_message_stories}")
print(f"  Single-message stories: {len(stories) - multi_message_stories}")
print(f"  Saved in {elapsed_segment:.1f}s")

In [ ]:
# Display the resulting stories.
story_df = pd.DataFrame([
    {
        "story_id": s.story_id[:12] + "...",
        "messages": len(s.message_ids),
        "start": s.timestamp_start.strftime("%Y-%m-%d %H:%M"),
        "end": s.timestamp_end.strftime("%Y-%m-%d %H:%M"),
        "text_preview": (s.english_combined_text or s.combined_text or ""),
    }
    for s in stories
])

# Sort the dataframe by the 'messages' column in descending order
story_df = story_df.sort_values(by="messages", ascending=False)

print(f"{len(stories)} stories from {len(raw_messages)} messages "
      f"(avg {len(raw_messages)/max(len(stories),1):.1f} msgs/story):")
display(story_df)

# Optional: Adding index=False prevents pandas from writing the row indices (which are now scrambled from sorting) to the CSV
story_df.to_csv("filename.csv", index=False)


73 stories from 100 messages (avg 1.4 msgs/story):


,story_id,messages,start,end,text_preview
0,867d5b39-ddf...,1,2026-04-13 11:10,2026-04-13 11:10,Iran warns any attack on its ports will make every Persian Gulf port insecure\n\nIran's military has warned that any threat to the count...
37,28d7aeb1-1c5...,1,2026-04-13 17:00,2026-04-13 17:00,🔴 Iran’s IRGC Ground Force commander Brigadier General Mohammad Karami:\n\n🔺 Iran’s armed forces have established a multi-layered securi...
53,2fd5ef59-af8...,1,2026-04-13 19:29,2026-04-13 19:29,"Member of the martyred leader's office, Mehdi Fazaeli, says the US and Israel thought that by assassinating the Leader of Iran the count..."
52,3fbfc5a3-610...,1,2026-04-13 19:29,2026-04-13 19:29,Prominent Iranian cleric Ayatollah Nouri Hamedani praised Pope Leo XIV in a letter for his stance against US-Israeli war crimes:\n\n🔺 Yo...
51,cd31830a-691...,1,2026-04-13 19:14,2026-04-13 19:14,Iran's Parliament Speaker Qalibaf hails Pope Leo XIV’s stance against the US-Israeli war crimes:\n\nHonoring Pope Leo’s fearless stand!\...
...,...,...,...,...,...
24,a3214a78-011...,1,2026-04-13 14:38,2026-04-13 14:38,"Reza Vedadi, researcher and author, explains how Trump would implement his naval blockade against Iran.\n\n@PressTV"
23,ea0f696d-ef2...,1,2026-04-13 14:20,2026-04-13 14:20,Pope Leo pledges to continue to speak out against war after Trump attack\n\nPope Leo XIV says he will continue to speak out against war ...
22,cbf30cdc-dc0...,1,2026-04-13 14:07,2026-04-13 14:07,Yemeni artist Kamal Sharaf's new cartoon on Trump's Iran naval blockade threat\n\n@PressTV
21,ab6e2658-2f7...,1,2026-04-13 14:00,2026-04-13 14:00,"People from Qom, central Iran, held a 25-km car convoy to Jamkaran Mosque to reaffirm their allegiance to Ayatollah Seyed Mojtaba Khamen..."


---
## 6. Story Embeddings

Each story's text is embedded via OpenAI (`text-embedding-3-small` by default, 1536 dimensions)
and upserted into the **Pinecone story index**. These embeddings serve two purposes:

1. **Cross-channel matching** — finding the same event reported by different channels.
2. **Semantic search** — querying stories by meaning rather than keywords.

In [ ]:
from telegram_scraper.kg.models import StoryEmbeddingRecord

t0 = time.time()
story_texts = [
    safe_story_text(
        (preferred_story_text(s) or "(media only)").strip(),
        max_chars=kg_settings.semantic_max_chars,
    ) or "(media only telegram story)"
    for s in stories
]
story_embeddings = embedder.embed_texts(story_texts) if story_texts else []

embedding_records = [
    StoryEmbeddingRecord(
        story_id=story.story_id,
        embedding=emb,
        channel_id=story.channel_id,
        timestamp_start=story.timestamp_start,
        node_ids=(),  # Will be updated after node resolution.
    )
    for story, emb in zip(stories, story_embeddings)
]
vector_store.upsert_story_embeddings(embedding_records)
elapsed_embed = time.time() - t0

dim = len(story_embeddings[0]) if story_embeddings else 0
print(f"Embedded {len(stories)} stories ({dim}-dim vectors) and upserted to Pinecone in {elapsed_embed:.1f}s")

---
## 7. Semantic Extraction (Preview)

The `OpenAISemanticExtractor` sends each story's text to the LLM with a structured prompt,
asking it to extract:

- **Events** — discrete occurrences (airstrikes, talks, launches)
- **People** — named individuals
- **Nations** — countries/nation-states
- **Organizations** — military, governmental, corporate entities
- **Places** — geographic locations
- **Themes** — abstract ongoing narratives (e.g., "Israel-Hamas conflict")
- **Primary event** — the single most important event in the story

Each entity includes: `name`, `summary`, `aliases`, `start_at`, `end_at`.

> This section previews extraction on the first 3 stories for observability.
> The full extraction for all stories runs inside `process_stories()` in the next section.

In [ ]:
# Preview extraction on a small subset to inspect the raw LLM output.
preview_count = min(3, len(stories))
t0 = time.time()
preview_extractions = extractor.extract_stories(stories[:preview_count])
elapsed_preview = time.time() - t0

print(f"Extracted semantics for {preview_count} stories in {elapsed_preview:.1f}s\n")

# Summary table: entity counts by kind per story.
extraction_df = pd.DataFrame([
    {
        "story_id": e.story_id[:12] + "...",
        "events": len(e.events),
        "people": len(e.people),
        "nations": len(e.nations),
        "orgs": len(e.orgs),
        "places": len(e.places),
        "themes": len(e.themes),
        "primary_event": (e.primary_event or "")[:60],
    }
    for e in preview_extractions
])
display(extraction_df)

# Aggregate totals.
totals = extraction_df[["events", "people", "nations", "orgs", "places", "themes"]].sum()
print(f"\nEntity totals across {preview_count} preview stories:")
display(totals.to_frame("count"))

In [ ]:
# Show the full extraction JSON for the first story.
from telegram_scraper.kg.node_resolver import serialize_extraction

if preview_extractions:
    sample_ext = preview_extractions[0]
    sample_story = stories[0]
    print(f"Full extraction for story {sample_ext.story_id[:20]}...")
    print(f"Story text: {(sample_story.english_combined_text or sample_story.combined_text or '')[:300]}\n")
    display(JSON(serialize_extraction(sample_ext)))

---
## 8. Node Resolution & Assignment

This is the core pipeline step. `KGNodeProcessingService.process_stories()` orchestrates:

1. **Extraction** — Calls the semantic extractor on all stories (including re-extracting the preview ones).
2. **Candidate cleaning** — Normalizes names, deduplicates aliases, canonicalizes event labels.
3. **Embedding** — Generates embeddings for event/theme candidates.
4. **Node resolution** — For each candidate, the `NodeResolver` either:
   - Matches to an existing node (by exact name, alias, or centroid similarity), or
   - Creates a new node (as `active` or `candidate` status based on promotion rules).
5. **Assignment** — Creates `StoryNodeAssignment` records linking stories to nodes with confidence scores.
6. **Cross-channel matching** — Finds stories across channels reporting the same event.
7. **Projection** — Rebuilds node relations and refreshes heat metrics.
8. **Event hierarchy** — Groups related events into parent-child structures.

We use this high-level service because the `NodeResolver` state machine is complex
and stateful — decomposing it would mean reimplementing ~200 lines of orchestration logic.

In [ ]:
from telegram_scraper.kg.services import (
    KGNodeProcessingService,
    KGNodeProjectionService,
    KGProcessingProgress,
)
from telegram_scraper.kg.event_hierarchy import KGEventHierarchyService

# Build the services. We hold references to projection_service and hierarchy_service
# so we can also call them independently in later sections.
projection_service = KGNodeProjectionService(
    repository=repository,
    vector_store=vector_store,
)
hierarchy_service = KGEventHierarchyService(repository)

node_service = KGNodeProcessingService(
    repository=repository,
    vector_store=vector_store,
    embedder=embedder,
    extractor=extractor,
    settings=kg_settings,
    projection_service=projection_service,
    hierarchy_service=hierarchy_service,
)

# Progress callback prints inline updates.
def _progress(p: KGProcessingProgress):
    print(
        f"  [{p.stories_processed}/{p.stories_total}] "
        f"nodes={p.nodes_created} assignments={p.assignments_created} "
        f"failures={p.failures} rate={p.rate_per_sec:.1f}/s"
    )

t0 = time.time()
process_result = node_service.process_stories(
    stories,
    progress_callback=_progress,
)
elapsed_process = time.time() - t0

print(f"\nNode processing complete in {elapsed_process:.1f}s:")
print(f"  Assignments created:    {process_result.assignments_created}")
print(f"  Nodes created:          {process_result.nodes_created}")
print(f"  Cross-channel matches:  {process_result.cross_channel_matches}")
print(f"  Relations created:      {process_result.relations_created}")
print(f"  Theme stats written:    {process_result.theme_stats_written}")

In [ ]:
# Display all nodes in the graph, sorted by article count.
all_nodes = repository.list_nodes(status=None)
node_df = pd.DataFrame([
    {
        "node_id": n.node_id[:12] + "...",
        "kind": n.kind,
        "slug": n.slug,
        "display_name": n.display_name,
        "status": n.status,
        "articles": n.article_count,
    }
    for n in sorted(all_nodes, key=lambda n: (-n.article_count, n.kind))
])

active_count = sum(1 for n in all_nodes if n.status == "active")
candidate_count = sum(1 for n in all_nodes if n.status == "candidate")
print(f"Total nodes: {len(all_nodes)} (active={active_count}, candidate={candidate_count})")
print(f"\nBy kind:")
display(node_df["kind"].value_counts().to_frame("count"))
print(f"\nTop 50 nodes:")
display(node_df.head(50))

In [ ]:
# Show the node assignments for the first story — which entities does the graph
# associate with this story, and with what confidence?
sample_story = stories[0]
assignments = repository.get_story_node_assignments(sample_story.story_id)
assigned_node_ids = [a.node_id for a in assignments]
nodes_lookup = {
    n.node_id: n
    for n in repository.get_nodes(assigned_node_ids)
} if assigned_node_ids else {}

assign_df = pd.DataFrame([
    {
        "kind": nodes_lookup[a.node_id].kind if a.node_id in nodes_lookup else "?",
        "display_name": nodes_lookup[a.node_id].display_name if a.node_id in nodes_lookup else "?",
        "confidence": f"{a.confidence:.2f}",
        "is_primary_event": a.is_primary_event,
    }
    for a in assignments
])

story_text = (sample_story.english_combined_text or sample_story.combined_text or "")[:300]
print(f"Assignments for story {sample_story.story_id[:20]}...")
print(f"Text: {story_text}\n")
display(assign_df)

---
## 9. Projection (Relations & Theme Stats)

`KGNodeProjectionService.refresh_all()` rebuilds two derived datasets:

1. **Node relations** — Co-occurrence edges between nodes (shared stories, cosine similarity).
   If two nodes appear together in many stories, they get a strong relation.
2. **Theme daily stats** — For each theme, computes daily `article_count` and `centroid_drift`
   (how much the theme's meaning shifted compared to the previous day).

> Note: `process_stories()` already ran this in Section 8 (with `projection_policy='per_batch'`).
> We run it again here to demonstrate it as a standalone step and pick up any additional data.

In [ ]:
t0 = time.time()
projection_result = projection_service.refresh_all(days=31)
elapsed_projection = time.time() - t0

print(f"Projection refresh complete in {elapsed_projection:.1f}s:")
print(f"  Relations created:     {projection_result.relations_created}")
print(f"  Theme stats written:   {projection_result.theme_stats_written}")

# Show a sample of relations for the most prominent node.
if all_nodes:
    top_node = sorted(all_nodes, key=lambda n: -n.article_count)[0]
    relations = repository.list_node_relations(top_node.node_id)
    if relations:
        related_ids = [r.target_node_id if r.source_node_id == top_node.node_id else r.source_node_id for r in relations]
        related_lookup = {n.node_id: n for n in repository.get_nodes(related_ids)}
        rel_df = pd.DataFrame([
            {
                "related_to": related_lookup[rid].display_name if rid in related_lookup else "?",
                "kind": related_lookup[rid].kind if rid in related_lookup else "?",
                "score": f"{r.score:.3f}",
                "shared_stories": r.shared_story_count,
            }
            for r, rid in zip(relations, related_ids)
        ])
        print(f"\nRelations for '{top_node.display_name}' ({top_node.kind}):")
        display(rel_df.head(15))

---
## 10. Event Hierarchy

`KGEventHierarchyService.rebuild()` organizes event nodes into parent-child trees:

- **Explicit parents**: Named operations (e.g., "Operation True Promise") become parents
  of their sub-events.
- **Generic families**: Groups like "Israeli airstrikes" or "Launches toward Israel" are
  synthesized from event name patterns (actor + family + place).
- **Child events**: Individual occurrences linked via `parent_node_id`.

This reduces noise in the graph by rolling up repetitive events under meaningful parents.

In [ ]:
t0 = time.time()
hierarchy_result = hierarchy_service.rebuild()
elapsed_hierarchy = time.time() - t0

print(f"Event hierarchy rebuilt in {elapsed_hierarchy:.1f}s:")
print(f"  Parents created:      {hierarchy_result.parents_created}")
print(f"  Parents deleted:      {hierarchy_result.parents_deleted}")
print(f"  Child links updated:  {hierarchy_result.child_links_updated}")
print(f"  Top-level events:     {hierarchy_result.top_level_events}")

---
## 11. Heat & Phase Classification

The **node heat view** is a materialized Postgres view that computes normalized article counts
per time window (1d, 3d, 5d, 7d, 14d, 31d) for each node.

The `classify_phase()` function then categorizes each node into a lifecycle phase:

| Phase | Meaning | Condition |
|-------|---------|----------|
| **emerging** | New, sharp spike | `heat_1d > 0.10` and `heat_31d < 0.02` |
| **fading** | Declining topic | `heat_31d > 0.05` and `heat_1d < 0.01` |
| **sustained** | Steady presence | `\|heat_1d - heat_31d\| < 0.02` |
| **flash_event** | Brief spike | `heat_3d > 0.10` and `heat_7d < 0.02` |
| **steady** | Default fallback | — |

In [ ]:
from telegram_scraper.kg.heat_phase import classify_phase

# Refresh the materialized view to include newly ingested data.
t0 = time.time()
repository.refresh_node_heat_view()
elapsed_heat = time.time() - t0
print(f"Node heat view refreshed in {elapsed_heat:.1f}s")

# Load heat snapshots for themes and events.
theme_heat = repository.list_node_heat_rows(kind="theme")
event_heat = repository.list_node_heat_rows(kind="event")

# Classify phases and build a combined DataFrame.
heat_rows = []
for snapshot in theme_heat:
    phase = classify_phase(snapshot, kg_settings.theme_heat_thresholds)
    heat_rows.append({
        "kind": snapshot.kind,
        "display_name": snapshot.display_name,
        "articles": snapshot.article_count,
        "heat_1d": f"{snapshot.heat_1d:.4f}",
        "heat_7d": f"{snapshot.heat_7d:.4f}",
        "heat_31d": f"{snapshot.heat_31d:.4f}",
        "phase": phase,
    })
for snapshot in event_heat:
    phase = classify_phase(snapshot, kg_settings.event_heat_thresholds)
    heat_rows.append({
        "kind": snapshot.kind,
        "display_name": snapshot.display_name,
        "articles": snapshot.article_count,
        "heat_1d": f"{snapshot.heat_1d:.4f}",
        "heat_7d": f"{snapshot.heat_7d:.4f}",
        "heat_31d": f"{snapshot.heat_31d:.4f}",
        "phase": phase,
    })

if heat_rows:
    heat_df = pd.DataFrame(heat_rows)
    print(f"\n{len(theme_heat)} themes + {len(event_heat)} events with heat data:")
    print(f"\nPhase distribution:")
    display(heat_df["phase"].value_counts().to_frame("count"))
    print(f"\nTop 20 by article count:")
    display(heat_df.sort_values("articles", ascending=False).head(20))
else:
    print("No heat data available yet. Heat requires data spanning multiple days.")

---
## 12. Query & Inspect

Use `KGQueryService` to query the knowledge graph through the same interface
that powers the visualization API. This is a read-only inspection of the final state.

In [ ]:
from telegram_scraper.kg.services import KGQueryService

query_service = KGQueryService(repository)

# Channels.
print("=" * 60)
print("CHANNELS")
print("=" * 60)
for ch in query_service.channels():
    print(f"  {ch.channel_title} (id={ch.channel_id}, stories={ch.story_count})")

# Top events.
print(f"\n{'=' * 60}")
print("TOP EVENTS")
print("=" * 60)
events = query_service.list_nodes(kind="event", limit=15)
if events:
    event_list_df = pd.DataFrame([
        {
            "slug": e.slug,
            "display_name": e.display_name,
            "articles": e.article_count,
            "children": e.child_count,
        }
        for e in events
    ])
    display(event_list_df)
else:
    print("  (no events)")

# Top themes.
print(f"\n{'=' * 60}")
print("TOP THEMES")
print("=" * 60)
themes = query_service.list_nodes(kind="theme", limit=15)
if themes:
    theme_list_df = pd.DataFrame([
        {
            "slug": t.slug,
            "display_name": t.display_name,
            "articles": t.article_count,
        }
        for t in themes
    ])
    display(theme_list_df)
else:
    print("  (no themes)")

# Top people.
print(f"\n{'=' * 60}")
print("TOP PEOPLE")
print("=" * 60)
people = query_service.list_nodes(kind="person", limit=10)
if people:
    people_df = pd.DataFrame([
        {"slug": p.slug, "display_name": p.display_name, "articles": p.article_count}
        for p in people
    ])
    display(people_df)
else:
    print("  (no people)")

In [ ]:
# Drill into a specific node to see its full detail: related entities and sample stories.
# Change NODE_KIND and NODE_SLUG to inspect different nodes.

NODE_KIND = "event"  # Options: event, theme, person, nation, org, place
NODE_SLUG = events[0].slug if events else ""  # Pick the top event by default.

if NODE_SLUG:
    detail = query_service.node_show(kind=NODE_KIND, slug=NODE_SLUG, story_limit=5)
    if detail:
        print(f"{'=' * 60}")
        print(f"{detail.display_name} ({detail.kind})")
        print(f"{'=' * 60}")
        print(f"  Summary:    {detail.summary or '(none)'}")
        print(f"  Articles:   {detail.article_count}")
        if detail.parent_event:
            print(f"  Parent:     {detail.parent_event.slug}")
        if detail.child_events:
            print(f"  Children:   {len(detail.child_events)}")
            for child in detail.child_events[:5]:
                print(f"    - {child.display_name} ({child.article_count} articles)")

        # Related entities.
        for label, related in [
            ("People", detail.people),
            ("Nations", detail.nations),
            ("Places", detail.places),
            ("Orgs", detail.orgs),
            ("Themes", detail.themes),
        ]:
            if related:
                names = [r.display_name for r in related[:5]]
                print(f"  {label}:    {', '.join(names)}")

        # Sample stories.
        if detail.stories:
            print(f"\n  --- Sample stories ({len(detail.stories)} shown) ---")
            for story in detail.stories:
                date_str = story.timestamp_start.strftime("%Y-%m-%d %H:%M")
                print(f"  [{date_str}] ({story.channel_title}) {story.preview_text[:200]}")
    else:
        print(f"Node not found: {NODE_KIND}/{NODE_SLUG}")
else:
    print("No nodes available to inspect. Run with more messages or check extraction output.")

---
## 13. Pipeline Summary

Cumulative statistics for the entire pipeline run.

In [ ]:
actually_translated_count = sum(
    1 for m in translated_messages
    if m.source_language not in ("en", "und", None)
)

print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"  Channel:                {selected_chat.title} (id={selected_chat.chat_id})")
print(f"  Messages ingested:      {len(raw_messages)}")
print(f"  Messages translated:    {actually_translated_count}")
print(f"  Stories created:        {len(stories)}")
print(f"  Nodes created:          {process_result.nodes_created}")
print(f"  Assignments created:    {process_result.assignments_created}")
print(f"  Cross-channel matches:  {process_result.cross_channel_matches}")
print(f"  Relations created:      {process_result.relations_created}")
print(f"  Theme stats written:    {process_result.theme_stats_written}")
print(f"  Hierarchy parents:      {hierarchy_result.parents_created}")
print(f"  Top-level events:       {hierarchy_result.top_level_events}")
print(f"  Total nodes in graph:   {len(all_nodes)}")
print(f"\nTiming:")
print(f"  Fetch messages:     {elapsed_fetch:.1f}s")
print(f"  Translation:        {elapsed_translate:.1f}s")
print(f"  Segmentation:       {elapsed_segment:.1f}s")
print(f"  Story embeddings:   {elapsed_embed:.1f}s")
print(f"  Node processing:    {elapsed_process:.1f}s")
print(f"  Projection:         {elapsed_projection:.1f}s")
print(f"  Hierarchy:          {elapsed_hierarchy:.1f}s")
print(f"  Heat refresh:       {elapsed_heat:.1f}s")
total_elapsed = (
    elapsed_fetch + elapsed_translate + elapsed_segment + elapsed_embed
    + elapsed_process + elapsed_projection + elapsed_hierarchy + elapsed_heat
)
print(f"  ----------------------")
print(f"  Total pipeline:     {total_elapsed:.1f}s")

---
## 14. Cleanup (Optional)

Disconnect the Telegram client. Optionally reset the channel's semantic state if you
want to re-run the pipeline from scratch on the same channel.

In [ ]:
# Disconnect the Telegram client.
await telegram_client.disconnect()
print("Telegram client disconnected.")

In [ ]:
# Optional: Reset all semantic data for the selected channel.
# This deletes nodes, assignments, and story semantics — but preserves raw messages and stories.
# Set RESET_CHANNEL = True to execute.

RESET_CHANNEL = False

if RESET_CHANNEL:
    deleted_node_ids, deleted_assignment_story_ids, deleted_semantic_story_ids = (
        repository.clear_semantic_state(channel_id=selected_chat.chat_id)
    )
    print(f"Semantic state cleared for channel {selected_chat.chat_id}:")
    print(f"  Nodes deleted:              {len(deleted_node_ids)}")
    print(f"  Assignment stories cleared: {len(deleted_assignment_story_ids)}")
    print(f"  Semantic records cleared:   {len(deleted_semantic_story_ids)}")
else:
    print("RESET_CHANNEL is False — no cleanup performed. Set to True and re-run to reset.")